In [25]:
import pandas as pd
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [26]:
df_init = pd.read_excel("/kaggle/input/datasets/impuneetmishra/ct-data/preprocessed_patient_data_existingCTs.xlsx", sheet_name='Sheet1')
df_fu = pd.read_excel("/kaggle/input/datasets/impuneetmishra/ct-data/preprocessed_BUCS_patient_data_followUp.xlsx", sheet_name='Sheet1')

In [55]:
df_init.columns.tolist()

['Unnamed: 0',
 'ID',
 'KeptInTrial',
 'KeepinAnalysis',
 'lesion PR',
 'GM quality',
 'WM',
 'CT date (MM/DD/YYYYY)',
 'Pathname',
 'Date of entering the pool',
 'Type of stroke',
 'Scan – Date',
 'F15age',
 'F16gend',
 'F17hand',
 'F18educ',
 'year',
 'F19ethn',
 'F2histo',
 'F31stro',
 'F32hosp',
 'F34type',
 'F35scan',
 'scanday',
 'F35date',
 'F37les',
 'F38vasc',
 'F38spec',
 'F39side',
 'F310loc',
 'spec. cort',
 'spec.cort B',
 'F4set1',
 'F42Examiner',
 'F43-Site',
 'F45scre',
 'Time (BcoS – Stroke)days',
 'Barthel',
 'Anxiety',
 'Depress',
 'N1Mpi',
 'S1Mpi',
 'N1MtsF',
 'S1MtsF',
 'N1MtsM',
 'S1MtsM',
 'N1Mno',
 'S1Mno',
 'N2Lpn',
 'S2Lpn',
 'N3Lsc',
 'S3Lsc',
 'N4Lsr',
 'S4LsrA',
 'S4LsrT',
 'N5Lre',
 'S5LreA',
 'S5LreT',
 'N61Mfi',
 'S61Mfi',
 'N62Mri',
 'S62Mri',
 'N7Akc',
 'S7AkcA',
 'S7AkcF',
 'S7AkcS',
 'N8Ave',
 'S8AveLU',
 'S8AveRU',
 'S8AveLB',
 'S8AveRB',
 'N9Ate',
 'S9AteLU',
 'S9AteRU',
 'S9AteLB',
 'S9AteRB',
 'N10Abf',
 'S10AbfA',
 'S10AbfR',
 'N11Aaa',
 'S11Aa

### Data Preparation

In [27]:
import pandas as pd
import numpy as np
import datetime
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor

# 1. Merge baseline features with follow-up target
df_fu_target = df_fu[['ID', 'S2Lpn_fu']].copy()
df_merged = pd.merge(df_init, df_fu_target, on='ID', how='inner')

# 2. Preprocess & Impute (Leak-Proof)
def preprocess_prognostic(df):
    if 'Unnamed: 0' in df.columns:
        df = df.rename(columns={'Unnamed: 0': 'idx'})
        
    df = df.dropna(subset=['S2Lpn_fu'])
    
    cols = ['F2histo', 'F37les', 'F310loc', 'F45scre', 'Barthel']
    cols_present = [c for c in cols if c in df.columns]
    mask = df[cols_present].map(lambda x: isinstance(x, str)).any(axis=1)
    df = df[~mask]
    
    cols_with_dt = [c for c in df.columns if df[c].dtype != 'datetime64[ns]' and df[c].apply(lambda x: isinstance(x, (datetime.datetime, datetime.date))).any()]
    for c in cols_with_dt:
        df[c] = pd.to_datetime(df[c], errors='coerce')
        
    df_numeric = df.select_dtypes(include=['number'])
    
    empty_cols = df_numeric.columns[df_numeric.isna().all()].tolist()
    if empty_cols:
        print(f"Purging 100% empty columns: {empty_cols}")
        df_numeric = df_numeric.drop(columns=empty_cols)
    
    # CRITICAL: Isolate target to prevent it from influencing baseline imputation
    target_series = df_numeric['S2Lpn_fu'].copy()
    df_impute_features = df_numeric.drop(columns=['S2Lpn_fu'])
    
    print("Executing Leak-Proof MICE Imputation via Random Forest...")
    mice_imputer = IterativeImputer(
        estimator=RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1),
        max_iter=10,
        random_state=42
    )
    
    imputed_array = mice_imputer.fit_transform(df_impute_features)
    df_imputed = pd.DataFrame(
        imputed_array, 
        columns=df_impute_features.columns, 
        index=df_impute_features.index
    )
    
    # Reattach target post-imputation
    df_imputed['S2Lpn_fu'] = target_series.values
    
    return df_imputed

df_clean = preprocess_prognostic(df_merged)

# 3. Extract Arrays & Align Indices
valid_indices = df_clean['idx'].values.astype(int)
drop_cols = ['S2Lpn_fu', 'S2Lpn', 'ID', 'idx']
X_tab = df_clean.drop(columns=[c for c in drop_cols if c in df_clean.columns]).values
y_target = df_clean['S2Lpn_fu'].values

Purging 100% empty columns: ['scanday']
Executing Leak-Proof MICE Imputation via Random Forest...


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


### Three-way split and ResNet feature extraction

In [29]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import TensorDataset, DataLoader

# 1. Load and Standardize CT Scans
# Verify the string path matches your BEAR/Kaggle directory
ct_array = np.load("/kaggle/input/datasets/impuneetmishra/ct-data/stitched_CT_scans_BCoS.npy") 
ct_matched = ct_array[valid_indices]
ct_tensor = torch.tensor(ct_matched, dtype=torch.float32).unsqueeze(1).repeat(1, 3, 1, 1)
ct_tensor = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])(ct_tensor)

# 2. Architect Three-Way Split (60% CNN Train, 20% Ridge Tune, 20% Test)
idx_temp, idx_test = train_test_split(np.arange(len(y_target)), test_size=0.2, random_state=42)
idx_cnn, idx_ridge = train_test_split(idx_temp, test_size=0.25, random_state=42)

tensor_cnn, y_cnn = ct_tensor[idx_cnn], y_target[idx_cnn]
tensor_ridge, y_ridge = ct_tensor[idx_ridge], y_target[idx_ridge]
tensor_test, y_test = ct_tensor[idx_test], y_target[idx_test]

X_tab_ridge, X_tab_test = X_tab[idx_ridge], X_tab[idx_test]

# 3. Optimize ResNet-18 (Split 1)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
resnet_custom = models.resnet18(weights=None)
resnet_custom.fc = nn.Linear(resnet_custom.fc.in_features, 1)
resnet_custom = resnet_custom.to(device)

optimizer = optim.Adam(resnet_custom.parameters(), lr=1e-4, weight_decay=1e-2)
criterion = nn.MSELoss()
train_dl = DataLoader(TensorDataset(tensor_cnn, torch.tensor(y_cnn, dtype=torch.float32).unsqueeze(1)), batch_size=32, shuffle=True)

resnet_custom.train()
for epoch in range(15):
    for batch_x, batch_y in train_dl:
        optimizer.zero_grad()
        loss = criterion(resnet_custom(batch_x.to(device)), batch_y.to(device))
        loss.backward()
        optimizer.step()

# 4. Extract Static Feature Volumes (Splits 2 & 3)
extractor = nn.Sequential(*list(resnet_custom.children())[:-1]).eval()
with torch.no_grad():
    cnn_feat_ridge = extractor(tensor_ridge.to(device)).view(len(tensor_ridge), -1).cpu().numpy()
    cnn_feat_test = extractor(tensor_test.to(device)).view(len(tensor_test), -1).cpu().numpy()

### Scaling & Dimensionality Compression

In [30]:
from sklearn.decomposition import PCA

# 1. Scale Modalities
scaler_tab = StandardScaler()
tab_ridge_scaled = scaler_tab.fit_transform(X_tab_ridge)
tab_test_scaled = scaler_tab.transform(X_tab_test)

scaler_cnn = StandardScaler()
cnn_ridge_scaled = scaler_cnn.fit_transform(cnn_feat_ridge)
cnn_test_scaled = scaler_cnn.transform(cnn_feat_test)

# 2. PCA Subspaces (95% Variance)
pca_cnn = PCA(n_components=0.95, random_state=42)
cnn_ridge_pca = pca_cnn.fit_transform(cnn_ridge_scaled)
cnn_test_pca = pca_cnn.transform(cnn_test_scaled)

pca_tab = PCA(n_components=0.95, random_state=42)
tab_ridge_pca = pca_tab.fit_transform(tab_ridge_scaled)
tab_test_pca = pca_tab.transform(tab_test_scaled)

print(f"CNN Dimensionality: {cnn_ridge_scaled.shape[1]} -> {cnn_ridge_pca.shape[1]} retained")
print(f"Tabular Dimensionality: {tab_ridge_scaled.shape[1]} -> {tab_ridge_pca.shape[1]} retained")

CNN Dimensionality: 512 -> 17 retained
Tabular Dimensionality: 84 -> 17 retained


### Experiment 1 - Tabular Only

In [31]:
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score, mean_absolute_error

# Logarithmic search space for L2 Penalty
alphas = np.logspace(-3, 4, 100)

# CV=5 internally cross-validates across Split 2 to find optimal alpha
model_tab = RidgeCV(alphas=alphas, cv=5)
model_tab.fit(tab_ridge_scaled, y_ridge)
preds_tab = model_tab.predict(tab_test_scaled)

print(f"Optimal Alpha: {model_tab.alpha_:.4f}")
print(f"Exp 1 | Tabular Only R2: {r2_score(y_test, preds_tab):.4f}")
print(f"Exp 1 | Tabular Only MAE: {mean_absolute_error(y_test, preds_tab):.4f}")

Optimal Alpha: 46.4159
Exp 1 | Tabular Only R2: -0.4864
Exp 1 | Tabular Only MAE: 2.1653


### Experiment 2 - ResNet Only (512D)

In [32]:
model_cnn = RidgeCV(alphas=alphas, cv=5)
model_cnn.fit(cnn_ridge_scaled, y_ridge)
preds_cnn = model_cnn.predict(cnn_test_scaled)

print(f"Optimal Alpha: {model_cnn.alpha_:.4f}")
print(f"Exp 2 | ResNet Only R2: {r2_score(y_test, preds_cnn):.4f}")
print(f"Exp 2 | ResNet Only MAE: {mean_absolute_error(y_test, preds_cnn):.4f}")

Optimal Alpha: 533.6699
Exp 2 | ResNet Only R2: -0.0320
Exp 2 | ResNet Only MAE: 2.1819


### Experiment 3 - Unreduced Fusion

In [33]:

X_fuse_ridge_all = np.hstack((cnn_ridge_scaled, tab_ridge_scaled))
X_fuse_test_all = np.hstack((cnn_test_scaled, tab_test_scaled))

model_fuse_all = RidgeCV(alphas=alphas, cv=5)
model_fuse_all.fit(X_fuse_ridge_all, y_ridge)
preds_fuse_all = model_fuse_all.predict(X_fuse_test_all)

print(f"Optimal Alpha: {model_fuse_all.alpha_:.4f}")
print(f"Exp 3 | Unreduced Fusion R2: {r2_score(y_test, preds_fuse_all):.4f}")
print(f"Exp 3 | Unreduced Fusion MAE: {mean_absolute_error(y_test, preds_fuse_all):.4f}")

Optimal Alpha: 1.0975
Exp 3 | Unreduced Fusion R2: -0.4347
Exp 3 | Unreduced Fusion MAE: 2.3530


### Experiment 4 - Asymmetric PCA Fusion

In [34]:
X_fuse_ridge_pca1 = np.hstack((cnn_ridge_pca, tab_ridge_scaled))
X_fuse_test_pca1 = np.hstack((cnn_test_pca, tab_test_scaled))

model_fuse_pca1 = RidgeCV(alphas=alphas, cv=5)
model_fuse_pca1.fit(X_fuse_ridge_pca1, y_ridge)
preds_fuse_pca1 = model_fuse_pca1.predict(X_fuse_test_pca1)

print(f"Optimal Alpha: {model_fuse_pca1.alpha_:.4f}")
print(f"Exp 4 | PCA ResNet (95%) + All Tabular R2: {r2_score(y_test, preds_fuse_pca1):.4f}")
print(f"Exp 4 | PCA ResNet (95%) + All Tabular MAE: {mean_absolute_error(y_test, preds_fuse_pca1):.4f}")

Optimal Alpha: 123.2847
Exp 4 | PCA ResNet (95%) + All Tabular R2: -0.3292
Exp 4 | PCA ResNet (95%) + All Tabular MAE: 2.2321


### Experiment 5 - Dual PCA Fusion

In [35]:
X_fuse_ridge_pca2 = np.hstack((cnn_ridge_pca, tab_ridge_pca))
X_fuse_test_pca2 = np.hstack((cnn_test_pca, tab_test_pca))

model_fuse_pca2 = RidgeCV(alphas=alphas, cv=5)
model_fuse_pca2.fit(X_fuse_ridge_pca2, y_ridge)
preds_fuse_pca2 = model_fuse_pca2.predict(X_fuse_test_pca2)

print(f"Optimal Alpha: {model_fuse_pca2.alpha_:.4f}")
print(f"Exp 5 | Dual PCA (95%) Fusion R2: {r2_score(y_test, preds_fuse_pca2):.4f}")
print(f"Exp 5 | Dual PCA (95%) Fusion MAE: {mean_absolute_error(y_test, preds_fuse_pca2):.4f}")

Optimal Alpha: 123.2847
Exp 5 | Dual PCA (95%) Fusion R2: -0.3284
Exp 5 | Dual PCA (95%) Fusion MAE: 2.2396


### Standardized Coefficient Magnitude Analysis

In [36]:
def compute_coefficient_mass(model, cnn_dim, exp_name):
    # Extract learned L2 coefficients
    W = np.squeeze(model.coef_)
    
    # Isolate domains
    W_resnet = W[:cnn_dim]
    W_tabular = W[cnn_dim:]
    
    # Calculate absolute mass
    mass_resnet = np.sum(np.abs(W_resnet))
    mass_tabular = np.sum(np.abs(W_tabular))
    total_mass = mass_resnet + mass_tabular
    
    # Convert to relative percentages
    pct_resnet = (mass_resnet / total_mass) * 100 if total_mass > 0 else 0
    pct_tabular = (mass_tabular / total_mass) * 100 if total_mass > 0 else 0
    
    print(f"--- {exp_name} ---")
    print(f"ResNet Predictive Variance:  {pct_resnet:.2f}%")
    print(f"Tabular Predictive Variance: {pct_tabular:.2f}%\n")

# Map exact model variables outputted from the previous cells
experiments = [
    {"name": "Tabular Only", "model": model_tab, "cnn_dim": 0},
    {"name": "ResNet Only (512D)", "model": model_cnn, "cnn_dim": 512},
    {"name": "Unreduced Fusion: All ResNet (512D) + All Tabular", "model": model_fuse_all, "cnn_dim": 512},
    {"name": "Asymmetric Fusion: PCA ResNet (95%) + All Tabular", "model": model_fuse_pca1, "cnn_dim": cnn_ridge_pca.shape[1]},
    {"name": "Dual PCA Fusion (95%)", "model": model_fuse_pca2, "cnn_dim": cnn_ridge_pca.shape[1]}
]

# Execute iterative extraction
for exp in experiments:
    compute_coefficient_mass(exp["model"], exp["cnn_dim"], exp["name"])

--- Tabular Only ---
ResNet Predictive Variance:  0.00%
Tabular Predictive Variance: 100.00%

--- ResNet Only (512D) ---
ResNet Predictive Variance:  100.00%
Tabular Predictive Variance: 0.00%

--- Unreduced Fusion: All ResNet (512D) + All Tabular ---
ResNet Predictive Variance:  70.19%
Tabular Predictive Variance: 29.81%

--- Asymmetric Fusion: PCA ResNet (95%) + All Tabular ---
ResNet Predictive Variance:  21.66%
Tabular Predictive Variance: 78.34%

--- Dual PCA Fusion (95%) ---
ResNet Predictive Variance:  43.81%
Tabular Predictive Variance: 56.19%



### RESULTS

In [37]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score, mean_absolute_error
from IPython.display import display

def get_variance_pct(model, cnn_dim):
    # Extract coefficients and calculate absolute variance mass
    W = np.squeeze(model.coef_)
    W_resnet = W[:cnn_dim]
    W_tabular = W[cnn_dim:]
    
    mass_resnet = np.sum(np.abs(W_resnet))
    mass_tabular = np.sum(np.abs(W_tabular))
    total_mass = mass_resnet + mass_tabular
    
    pct_resnet = (mass_resnet / total_mass) * 100 if total_mass > 0 else 0
    pct_tabular = (mass_tabular / total_mass) * 100 if total_mass > 0 else 0
    
    return f"{pct_resnet:.2f}%", f"{pct_tabular:.2f}%"

# Aggregate all experiments using the active variables from previous cells
table_data = [
    {
        "Experiment Architecture": "1. Tabular Only (Baseline)",
        "R² Score": f"{r2_score(y_test, preds_tab):.4f}",
        "MAE": f"{mean_absolute_error(y_test, preds_tab):.4f}",
        "ResNet Variance": get_variance_pct(model_tab, 0)[0],
        "Tabular Variance": get_variance_pct(model_tab, 0)[1]
    },
    {
        "Experiment Architecture": "2. ResNet Only (512D)",
        "R² Score": f"{r2_score(y_test, preds_cnn):.4f}",
        "MAE": f"{mean_absolute_error(y_test, preds_cnn):.4f}",
        "ResNet Variance": get_variance_pct(model_cnn, 512)[0],
        "Tabular Variance": get_variance_pct(model_cnn, 512)[1]
    },
    {
        "Experiment Architecture": "3. Unreduced Fusion (512D + Tabular)",
        "R² Score": f"{r2_score(y_test, preds_fuse_all):.4f}",
        "MAE": f"{mean_absolute_error(y_test, preds_fuse_all):.4f}",
        "ResNet Variance": get_variance_pct(model_fuse_all, 512)[0],
        "Tabular Variance": get_variance_pct(model_fuse_all, 512)[1]
    },
    {
        "Experiment Architecture": "4. Asymmetric PCA Fusion (PCA ResNet + Tabular)",
        "R² Score": f"{r2_score(y_test, preds_fuse_pca1):.4f}",
        "MAE": f"{mean_absolute_error(y_test, preds_fuse_pca1):.4f}",
        "ResNet Variance": get_variance_pct(model_fuse_pca1, cnn_ridge_pca.shape[1])[0],
        "Tabular Variance": get_variance_pct(model_fuse_pca1, cnn_ridge_pca.shape[1])[1]
    },
    {
        "Experiment Architecture": "5. Dual PCA Fusion (PCA ResNet + PCA Tabular)",
        "R² Score": f"{r2_score(y_test, preds_fuse_pca2):.4f}",
        "MAE": f"{mean_absolute_error(y_test, preds_fuse_pca2):.4f}",
        "ResNet Variance": get_variance_pct(model_fuse_pca2, cnn_ridge_pca.shape[1])[0],
        "Tabular Variance": get_variance_pct(model_fuse_pca2, cnn_ridge_pca.shape[1])[1]
    }
]

df_results = pd.DataFrame(table_data)

# Apply CSS styling for Jupyter rendering
styled_table = (
    df_results.style
    .set_caption("<b>Experimental Results: Modality Contribution & Predictive Performance</b>")
    .set_properties(**{'text-align': 'center', 'border': '1px solid black', 'padding': '8px'})
    .set_table_styles([
        {'selector': 'th', 'props': [('text-align', 'center'), ('background-color', '#f2f2f2'), ('border', '1px solid black'), ('font-weight', 'bold')]},
        {'selector': 'caption', 'props': [('font-size', '16px'), ('margin-bottom', '10px')]}
    ])
    .hide(axis='index')
)

display(styled_table)

Experiment Architecture,R² Score,MAE,ResNet Variance,Tabular Variance
1. Tabular Only (Baseline),-0.4864,2.1653,0.00%,100.00%
2. ResNet Only (512D),-0.0320,2.1819,100.00%,0.00%
3. Unreduced Fusion (512D + Tabular),-0.4347,2.3530,70.19%,29.81%
4. Asymmetric PCA Fusion (PCA ResNet + Tabular),-0.3292,2.2321,21.66%,78.34%
5. Dual PCA Fusion (PCA ResNet + PCA Tabular),-0.3284,2.2396,43.81%,56.19%


### Median based tabular preprocessing

In [44]:
import pandas as pd
import numpy as np
import datetime

# 1. Merge baseline features with follow-up target
df_fu_target = df_fu[['ID', 'S2Lpn_fu']].copy()
df_merged = pd.merge(df_init, df_fu_target, on='ID', how='inner')

# 2. Simplified Preprocessing (Median Imputation)
def preprocess_median(df):
    if 'Unnamed: 0' in df.columns:
        df = df.rename(columns={'Unnamed: 0': 'idx'})
        
    # Drop rows where the target is missing
    df = df.dropna(subset=['S2Lpn_fu'])
    
    # Strip string artefacts
    cols = ['F2histo', 'F37les', 'F310loc', 'F45scre', 'Barthel']
    cols_present = [c for c in cols if c in df.columns]
    mask = df[cols_present].map(lambda x: isinstance(x, str)).any(axis=1)
    df = df[~mask]
    
    # Drop datetimes to bypass imputer/regression failures
    cols_with_dt = [c for c in df.columns if df[c].dtype != 'datetime64[ns]' and df[c].apply(lambda x: isinstance(x, (datetime.datetime, datetime.date))).any()]
    df = df.drop(columns=cols_with_dt, errors='ignore')
    
    # Isolate strictly numeric modalities
    df_numeric = df.select_dtypes(include=['number'])
    
    # Purge completely empty columns to prevent shape mismatch
    df_numeric = df_numeric.dropna(axis=1, how='all')
    
    # Isolate target before imputation to prevent leakage
    target = df_numeric['S2Lpn_fu']
    features = df_numeric.drop(columns=['S2Lpn_fu'])
    
    # Impute missing feature values with the median of each respective column
    features_imputed = features.fillna(features.median())
    
    # Reattach target
    df_final = pd.concat([features_imputed, target], axis=1)
    
    return df_final

df_clean = preprocess_median(df_merged)

# 3. Extract Arrays
valid_indices = df_clean['idx'].values.astype(int)
drop_cols = ['S2Lpn_fu', 'idx', 'ID']

X_tab = df_clean.drop(columns=[c for c in drop_cols if c in df_clean.columns]).values
y_target = df_clean['S2Lpn_fu'].values

print(f"Tabular Preprocessing Complete.")
print(f"Patients Retained: {len(y_target)}")
print(f"Tabular Features:  {X_tab.shape[1]}")

Tabular Preprocessing Complete.
Patients Retained: 127
Tabular Features:  85


### Structural Alignment & ResNet Feature Extraction

In [45]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Force strict N-count alignment using the fresh tabular valid_indices
# Assumes ct_array is loaded in memory via np.load("stitched_CT_scans_BCoS.npy")
ct_matched = ct_array[valid_indices]

# 2. Build the image tensor to match the tabular rows
ct_tensor = torch.tensor(ct_matched, dtype=torch.float32).unsqueeze(1).repeat(1, 3, 1, 1)
ct_tensor = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])(ct_tensor)

# 3. Initialize headless ResNet
resnet_custom = models.resnet18(weights=None)
resnet_custom.fc = nn.Linear(resnet_custom.fc.in_features, 1)
extractor = nn.Sequential(*list(resnet_custom.children())[:-1]).eval()
extractor = extractor.to(device)

# 4. Extract spatial vectors for all valid patients
with torch.no_grad():
    cnn_feat_all = extractor(ct_tensor.to(device)).view(len(ct_tensor), -1).cpu().numpy()

print(f"ResNet Extraction Complete.")
print(f"ResNet Matrix Shape: {cnn_feat_all.shape}")

# Create master fusion array for the pipelines
X_fuse_raw = np.hstack((cnn_feat_all, X_tab))

ResNet Extraction Complete.
ResNet Matrix Shape: (127, 512)


### Nested CV Pipeline Execution

In [51]:
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error

# 1. Define Pipeline Architecture
cnn_dim = cnn_feat_all.shape[1]
tab_dim = X_tab.shape[1]

cnn_cols = list(range(cnn_dim))
tab_cols = list(range(cnn_dim, cnn_dim + tab_dim))

outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)
alphas = np.logspace(-3, 4, 100)

# Construct pipelines
pipe_tab = Pipeline([('scaler', StandardScaler()), ('ridge', RidgeCV(alphas=alphas))])
pipe_cnn = Pipeline([('scaler', StandardScaler()), ('ridge', RidgeCV(alphas=alphas))])
pipe_fuse_all = Pipeline([('scaler', StandardScaler()), ('ridge', RidgeCV(alphas=alphas))])

preprocessor_pca1 = ColumnTransformer([
    ('cnn_pca', Pipeline([('scaler', StandardScaler()), ('pca', PCA(n_components=0.95, random_state=42))]), cnn_cols),
    ('tab_scale', StandardScaler(), tab_cols)
])
pipe_pca1 = Pipeline([('prep', preprocessor_pca1), ('ridge', RidgeCV(alphas=alphas))])

preprocessor_pca2 = ColumnTransformer([
    ('cnn_pca', Pipeline([('scaler', StandardScaler()), ('pca', PCA(n_components=0.95, random_state=42))]), cnn_cols),
    ('tab_pca', Pipeline([('scaler', StandardScaler()), ('pca', PCA(n_components=0.95, random_state=42))]), tab_cols)
])
pipe_pca2 = Pipeline([('prep', preprocessor_pca2), ('ridge', RidgeCV(alphas=alphas))])

experiments = [
    {"name": "1. Tabular Only", "pipe": pipe_tab, "data": X_tab, "cnn_dim": 0},
    {"name": "2. ResNet Only", "pipe": pipe_cnn, "data": cnn_feat_all, "cnn_dim": 512},
    {"name": "3. Unreduced Fusion", "pipe": pipe_fuse_all, "data": X_fuse_raw, "cnn_dim": 512},
    {"name": "4. Asymmetric PCA Fusion", "pipe": pipe_pca1, "data": X_fuse_raw, "cnn_dim": "dynamic"},
    {"name": "5. Dual PCA Fusion", "pipe": pipe_pca2, "data": X_fuse_raw, "cnn_dim": "dynamic"}
]
from IPython.display import display

def compute_coefficient_mass(model, cnn_dim):
    """Returns (pct_resnet, pct_tabular) instead of printing."""
    W = np.squeeze(model.coef_)
    W_resnet = W[:cnn_dim]
    W_tabular = W[cnn_dim:]
    mass_resnet = np.sum(np.abs(W_resnet))
    mass_tabular = np.sum(np.abs(W_tabular))
    total_mass = mass_resnet + mass_tabular
    pct_resnet = (mass_resnet / total_mass) * 100 if total_mass > 0 else 0
    pct_tabular = (mass_tabular / total_mass) * 100 if total_mass > 0 else 0
    return pct_resnet, pct_tabular


# 2. Execute Nested CV + Coefficient Mass
results = []
print("Executing 5-Fold Nested CV...")

for exp in experiments:
    name = exp["name"]
    pipe = exp["pipe"]
    data = exp["data"]

    print(f"  Running: {name}...")

    # Honest out-of-fold predictions for scoring
    y_pred = cross_val_predict(pipe, data, y_target, cv=outer_cv, n_jobs=-1)
    r2 = r2_score(y_target, y_pred)
    mae = mean_absolute_error(y_target, y_pred)

    # Fit on ALL data — this is the "final chef" whose coefficients we inspect
    pipe.fit(data, y_target)
    final_ridge = pipe.named_steps['ridge']

    # For PCA pipelines, pull the REAL number of retained ResNet components
    if 'prep' in pipe.named_steps:
        cnn_dim_effective = (
            pipe.named_steps['prep']
            .named_transformers_['cnn_pca']
            .named_steps['pca']
            .n_components_
        )
    else:
        cnn_dim_effective = exp["cnn_dim"]

    pct_resnet, pct_tabular = compute_coefficient_mass(final_ridge, cnn_dim_effective)

    results.append({
        "Experiment": name,
        "R2": r2,
        "MAE": mae,
        "% ResNet": pct_resnet,
        "% Tabular": pct_tabular
    })

print("Done.\n")

# 3. Display Results as a Styled HTML Table
results_df = pd.DataFrame(results).reset_index(drop=True)

styled_results = (
    results_df.style
    .format({
        "R2": "{:.4f}",
        "MAE": "{:.4f}",
        "% ResNet": "{:.2f}%",
        "% Tabular": "{:.2f}%"
    })
    .background_gradient(subset=["R2"], cmap="Greens")
    .background_gradient(subset=["MAE"], cmap="Reds_r")
    .background_gradient(subset=["% ResNet"], cmap="Blues")
    .set_caption("Nested CV Results: Performance & Coefficient Mass")
    .set_table_styles([
        {"selector": "caption", "props": [("font-size", "16px"), ("font-weight", "bold"), ("padding", "10px")]},
        {"selector": "th", "props": [("background-color", "#333"), ("color", "white"), ("padding", "8px")]},
        {"selector": "td", "props": [("padding", "8px"), ("text-align", "center")]},
    ])
)

display(styled_results)

Executing 5-Fold Nested CV...
  Running: 1. Tabular Only...
  Running: 2. ResNet Only...
  Running: 3. Unreduced Fusion...
  Running: 4. Asymmetric PCA Fusion...
  Running: 5. Dual PCA Fusion...
Done.



,Experiment,R2,MAE,% ResNet,% Tabular
0,1. Tabular Only,0.3463,1.8290,0.00%,100.00%
1,2. ResNet Only,-0.0587,2.5766,100.00%,0.00%
2,3. Unreduced Fusion,0.3514,1.9778,64.83%,35.17%
3,4. Asymmetric PCA Fusion,0.3712,1.9261,34.51%,65.49%
4,5. Dual PCA Fusion,0.3624,1.9465,48.14%,51.86%


In [52]:
# RESULTS
import itertools
from scipy import stats
from sklearn.model_selection import cross_val_score

def corrected_paired_ttest(scores_a, scores_b, n_train, n_test):
    """
    Nadeau & Bengio corrected resampled t-test for paired CV fold scores.
    Accounts for the fact that CV folds share overlapping training data.
    """
    diff = np.array(scores_a) - np.array(scores_b)
    n = len(diff)
    mean_diff = np.mean(diff)
    var_diff = np.var(diff, ddof=1)

    correction = (1 / n) + (n_test / n_train)
    denom = np.sqrt(var_diff * correction)

    if denom == 0:
        t_stat = 0.0
    else:
        t_stat = mean_diff / denom

    dof = n - 1
    p_value = 2 * stats.t.sf(np.abs(t_stat), dof)  # two-tailed
    return t_stat, p_value, mean_diff


# 1. Collect per-fold R2 scores for each experiment (folds are aligned since outer_cv has fixed random_state)
fold_scores = {}
for exp in experiments:
    name = exp["name"]
    pipe = exp["pipe"]
    data = exp["data"]
    scores = cross_val_score(pipe, data, y_target, cv=outer_cv, scoring='r2', n_jobs=-1)
    fold_scores[name] = scores

# 2. Pairwise corrected tests across all experiment pairs
N = len(y_target)
n_folds = outer_cv.get_n_splits()
n_test = N / n_folds
n_train = N - n_test

pairwise_results = []
for name_a, name_b in itertools.combinations(fold_scores.keys(), 2):
    scores_a = fold_scores[name_a]
    scores_b = fold_scores[name_b]
    t_stat, p_val, mean_diff = corrected_paired_ttest(scores_a, scores_b, n_train, n_test)
    pairwise_results.append({
        "Comparison": f"{name_a}  vs  {name_b}",
        "Mean R2 Diff": mean_diff,
        "t-stat": t_stat,
        "p-value (raw)": p_val
    })

pairwise_df = pd.DataFrame(pairwise_results)

# 3. Holm-Bonferroni correction across all pairwise tests
pairwise_df = pairwise_df.sort_values("p-value (raw)").reset_index(drop=True)
m = len(pairwise_df)
pairwise_df["p-value (Holm-corrected)"] = [
    min(1.0, p * (m - i)) for i, p in enumerate(pairwise_df["p-value (raw)"])
]
# Enforce monotonicity (standard Holm step-up requirement)
pairwise_df["p-value (Holm-corrected)"] = pairwise_df["p-value (Holm-corrected)"].cummax()
pairwise_df["Significant (α=0.05)"] = pairwise_df["p-value (Holm-corrected)"] < 0.05

# 4. Display as styled HTML table
styled_pairwise = (
    pairwise_df.style
    .format({
        "Mean R2 Diff": "{:.4f}",
        "t-stat": "{:.3f}",
        "p-value (raw)": "{:.4f}",
        "p-value (Holm-corrected)": "{:.4f}"
    })
    .background_gradient(subset=["p-value (Holm-corrected)"], cmap="RdYlGn_r", vmin=0, vmax=1)
    .set_caption("Pairwise Significance: Nadeau-Bengio Corrected t-test (Holm-Bonferroni adjusted)")
    .set_table_styles([
        {"selector": "caption", "props": [("font-size", "16px"), ("font-weight", "bold"), ("padding", "10px")]},
        {"selector": "th", "props": [("background-color", "#333"), ("color", "white"), ("padding", "8px")]},
        {"selector": "td", "props": [("padding", "8px"), ("text-align", "center")]},
    ])
)

display(styled_pairwise)

,Comparison,Mean R2 Diff,t-stat,p-value (raw),p-value (Holm-corrected),Significant (α=0.05)
0,2. ResNet Only vs 4. Asymmetric PCA Fusion,-0.4200,-3.151,0.0345,0.3449,False
1,2. ResNet Only vs 3. Unreduced Fusion,-0.3863,-2.935,0.0426,0.3833,False
2,2. ResNet Only vs 5. Dual PCA Fusion,-0.4100,-2.914,0.0435,0.3833,False
3,3. Unreduced Fusion vs 4. Asymmetric PCA Fusion,-0.0337,-1.156,0.3122,1.0000,False
4,4. Asymmetric PCA Fusion vs 5. Dual PCA Fusion,0.0100,0.871,0.4330,1.0000,False
5,3. Unreduced Fusion vs 5. Dual PCA Fusion,-0.0237,-0.665,0.5426,1.0000,False
6,1. Tabular Only vs 2. ResNet Only,0.2489,0.613,0.5733,1.0000,False
7,1. Tabular Only vs 4. Asymmetric PCA Fusion,-0.1710,-0.360,0.7371,1.0000,False
8,1. Tabular Only vs 5. Dual PCA Fusion,-0.1610,-0.336,0.7535,1.0000,False
9,1. Tabular Only vs 3. Unreduced Fusion,-0.1373,-0.300,0.7791,1.0000,False
